In [7]:
import pandas as pd
import numpy as np
import ast
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim import corpora
from gensim.models import LdaModel
import os

# Download resources (only needed first time)
nltk.download('stopwords')
nltk.download('wordnet')

# === Load your CSV ===
df = pd.read_csv("labels_per_photo.csv")
print(df.columns)

# === Text preprocessing setup ===
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_labels(label_list):
    """
    Takes a list like ["dog park", "leash", "grass"]
    and returns cleaned tokens like ["dog_park", "leash", "grass"].
    """
    if not isinstance(label_list, list):
        # Safely evaluate from string representation
        try:
            label_list = ast.literal_eval(label_list)
        except Exception:
            return []

    cleaned = []
    for lbl in label_list:
        if not isinstance(lbl, str):
            continue

        # Remove non-letter characters (preserve internal spaces)
        lbl = re.sub(r'[^a-zA-Z\s]', '', lbl).lower()

        # Lemmatize each word and remove stopwords
        words = [lemmatizer.lemmatize(w) for w in lbl.split() if w not in stop_words and len(w) > 2]
        if not words:
            continue

        # Join multi-word labels with underscore to preserve as single token
        cleaned.append("_".join(words))

    return cleaned

# Apply cleaning
df["tokens"] = df["labels"].apply(clean_labels)

# === Folder setup ===
os.makedirs("lda_person_topics", exist_ok=True)

# === LDA per person ===
all_topic_results = {}

for person, subdf in df.groupby("person"):
    print(f"\n=== Running LDA for {person} ({len(subdf)} documents) ===")
    
    if len(subdf) < 3:
        print("Not enough documents for LDA, skipping.")
        continue
    
    # Create dictionary and corpus
    dictionary = corpora.Dictionary(subdf["tokens"])
    dictionary.filter_extremes(no_below=2, no_above=0.5)
    corpus = [dictionary.doc2bow(tokens) for tokens in subdf["tokens"]]

    num_topics = min(5, max(2, len(subdf)//5))

    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        random_state=42,
        passes=10
    )

    # === Extract topics ===
    topics_data = []
    for i, topic in lda_model.show_topics(num_topics=num_topics, formatted=False):
        words = [word for word, _ in topic]
        topics_data.append({
            "person": person,
            "topic_number": i,
            "top_words": ", ".join(words)
        })
        print(f"Topic {i}: {', '.join(words)}")

    # Save per-person topics
    topics_df = pd.DataFrame(topics_data)
    filename = f"lda_person_topics/{person}_topics.csv"
    topics_df.to_csv(filename, index=False)
    print(f"✅ Saved {filename}")

    all_topic_results[person] = {
        "lda_model": lda_model,
        "dictionary": dictionary,
        "topics_df": topics_df
    }

    # === Compute document-topic weights ===
    topic_weights = []
    for doc in corpus:
        doc_topics = lda_model.get_document_topics(doc, minimum_probability=0)
        topic_weights.append([weight for _, weight in doc_topics])
    
    topic_weights = np.array(topic_weights)
    topic_df = pd.DataFrame(topic_weights, columns=[f"Topic_{i}" for i in range(num_topics)])
    topic_df["person"] = person
    topic_df["photo_id"] = subdf.index

    for col in topic_df.columns:
        if col not in df.columns:
            df[col] = np.nan
    df.loc[subdf.index, [f"Topic_{i}" for i in range(num_topics)]] = topic_df[[f"Topic_{i}" for i in range(num_topics)]].values

# === Save combined output ===
df.to_csv("lda_results_per_person.csv", index=False)
print("\n✅ LDA complete — bigrams preserved exactly as given in label arrays.")
print("• Per-person topics → lda_person_topics/")
print("• Combined weights → lda_results_per_person.csv")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\micha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\micha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Index(['person', 'path', 'labels', 'indoor_outdoor', 'day_night',
       'season_hint', 'cat_outdoor_activities', 'cat_food_drink',
       'cat_cultural', 'cat_night_life', 'cat_concerts_shows',
       'cat_casino_gambling', 'cat_attractions', 'cat_sporting_events'],
      dtype='object')

=== Running LDA for Caro's Album (29 documents) ===
Topic 0: tropical_setting, sunbathing, relaxing, canvas, group_photo, urban_park, nature_view, indoor_exhibit, cultural_heritage, sunset
Topic 1: tree, nature, outdoor_adventure, river, architecture, historic_building, statue, rock_formation, nature_view, urban_park
Topic 2: sunset, indoor_exhibit, lake, calm_water, cultural_heritage, canvas, group_photo, relaxing, nature_view, urban_park
Topic 3: hiking_trail, outdoor_adventure, scenic_view, nature, canvas, urban_park, group_photo, nature_view, relaxing, sunset
Topic 4: group_photo, relaxing, cultural_heritage, clear_sky, canvas, tree, lake, calm_water, urban_park, sunbathing
✅ Saved lda_person_top